# Does preprocessing the spike waveforms improve the curation model? (fig5e, spontaneous regime)

Only two of the ten predictors depend on the waveforms:
* `silhouette`: PCA of the waveforms of the unit and its neighbours (PCA settings: 5 components, `by_channel_local`)
* `snr`: peak of the unit's template (average of its waveforms) over the noise level

The other predictors (`sd_ratio`, `mad_ratio`, `amplitude_cutoff`, firing and ISI metrics) come from spike amplitudes and spike times, which waveform preprocessing does not change. So each variant below replaces `silhouette` (and `snr` when the template changes) and the model is evaluated as in fig5e (100 train/test splits, 75/25, unpenalized GLM, threshold 0.8; same splits for all variants).

**variants**
* more spikes per unit (cap 500 -> 2000)
* SVD denoising of each channel's waveforms (rank 2, 3, 5)
* amplitude normalization (waveforms divided by the template peak: shape only)
* trough alignment of each spike
* main channels only (best 1, 3, 6 channels) for the PCA
* PCA settings: 3 or 10 components; `by_channel_global`; `concatenated`

**readout**: for each variant, how informative is the new feature on its own (area under the ROC curve for predicting good units), and the model's precision, recall and average precision (threshold-free), with paired differences to the baseline.

In [1]:
%load_ext autoreload
%autoreload 2
import os, shutil, warnings, time
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "1"
import numpy as np
import pandas as pd
import spikeinterface as si
import spikecurate
from spikeinterface import extract_waveforms
from spikeinterface.core import ChannelSparsity
from spikeinterface.postprocessing import compute_principal_components
from spikeinterface.qualitymetrics import compute_quality_metrics
from spikecurate.crossval import _split_train_test
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn import metrics as skm
warnings.filterwarnings("ignore")

os.chdir("/home/steeve/steeve/epfl/code/spikebias/")
import sys; sys.path.insert(0, os.getcwd())
from src.nodes.models.Flc import dataloader

WE_PATH = "dataset/01_intermediate/waveforms/npx_spont/SortingKS4_10m_single_units"
SORTING_TRUE_10mn = "dataset/00_raw/ground_truth_npx_spont_10m"
QUALITY_PATH = "dataset/01_intermediate/analysis/sorting_quality/sorting_quality.csv"
VIEW = "temp/improve_model_views/waveforms_view"      # symlink view without the extensions (nothing is written in WE_PATH)
JOB = dict(n_jobs=16, chunk_duration="1s", progress_bar=False)
LABEL = "quality_label"
SEEDS = np.arange(0, 100)

## Baseline dataset (10 quality metrics)

In [2]:
%%time
We = si.load_waveforms(WE_PATH, with_recording=True)
ids = np.array([int(u) for u in We.unit_ids])
good, bad = dataloader.get_good_and_bad_units(QUALITY_PATH, exp="NS", sorter="KS4")
data = spikecurate.load_dataset(ids, good, bad, We.sorting, si.load_extractor(SORTING_TRUE_10mn), We,
                                load_qm_if_exists=False, delta_time=1.3, job_kwargs=JOB)
dataset, predictors = data["dataset"], list(data["predictors"])
print(dataset.shape, predictors)

(184, 11) ['amplitude_cutoff', 'firing_range', 'firing_rate', 'isi_violations_ratio', 'rp_contamination', 'rp_violations', 'sd_ratio', 'snr', 'silhouette', 'mad_ratio']
CPU times: user 12.7 s, sys: 5.23 s, total: 18 s
Wall time: 1min 20s


## Waveform variants, run in parallel (one process per variant; the variants are in src/nodes/models/Flc/waveform_variants.py)

In [3]:

%%time
from joblib import Parallel, delayed
from src.nodes.models.Flc import waveform_variants as wv

# symlink view of the waveform folder without the extensions: nothing is written in WE_PATH
shutil.rmtree(VIEW, ignore_errors=True); os.makedirs(VIEW)
for f in os.listdir(WE_PATH):
    if f not in ("principal_components", "quality_metrics", "spike_amplitudes"):
        os.symlink(os.path.abspath(f"{WE_PATH}/{f}"), f"{VIEW}/{f}")

# the worker processes must be able to import the project
os.environ["PYTHONPATH"] = os.getcwd()
N_WORKERS = 12     # ~5 GB of waveforms per worker
def safe(name):
    try: return name, wv.compute_variant(name, VIEW)[1], None
    except Exception as e: return name, None, repr(e)[:200]
results = Parallel(n_jobs=N_WORKERS, backend="loky")(delayed(safe)(n) for n in wv.VARIANT_NAMES)
FEATS = {n: f for n, f, e in results if f is not None}
ERR = {n: e for n, f, e in results if e is not None}
print("computed:", len(FEATS), "| failed:", ERR)


computed: 13 | failed: {'PCA mode concatenated (6 main channels)': "IndexError('too many indices for array: array is 2-dimensional, but 3 were indexed')"}
CPU times: user 2.43 s, sys: 1.38 s, total: 3.81 s
Wall time: 9min 29s


## Effect on the classifier

In [4]:
def evaluate(df, cols):
    """precision / recall (threshold 0.8) and average precision, on the same 100 splits for every variant"""
    out = []
    for s in SEEDS:
        tr, te = _split_train_test(df, 0.75, s)
        Xtr, Xte = tr[cols], te[cols]
        mtr, mte = ~Xtr.isna().any(axis=1).values, ~Xte.isna().any(axis=1).values
        sc = StandardScaler().fit(Xtr[mtr]); Z = np.zeros(Xte.shape); Z[mte] = sc.transform(Xte[mte])
        clf = LogisticRegression(penalty=None, max_iter=5000).fit(sc.transform(Xtr[mtr]), tr[LABEL][mtr])
        p = clf.predict_proba(Z)[:, 1]; p[~mte] = 0
        pred = (p >= 0.8).astype(int)
        out.append((skm.precision_score(te[LABEL], pred, zero_division=0), skm.recall_score(te[LABEL], pred, zero_division=0),
                    skm.average_precision_score(te[LABEL], p)))
    return np.array(out)

ci = lambda x: 1.96 * np.std(x) / np.sqrt(len(x))
base = evaluate(dataset, predictors)
rows = [dict(variant="baseline (current features)", corr_silhouette=1.0,
             auc_silhouette=skm.roc_auc_score(dataset[LABEL][dataset.silhouette.notna()], dataset.silhouette.dropna()),
             precision=np.median(base[:, 0]), recall=np.median(base[:, 1]), AP=np.median(base[:, 2]),
             dP=0, dP_ci=0, dR=0, dR_ci=0, dAP=0, dAP_ci=0)]
for name, f in FEATS.items():
    d = dataset.copy()
    d["silhouette"] = f["silhouette"].reindex(d.index).values
    if "snr" in f: d["snr"] = f["snr"].reindex(d.index).values
    r = evaluate(d, predictors); dd = r - base
    ok = d.silhouette.notna()
    rows.append(dict(variant=name, corr_silhouette=np.corrcoef(d.silhouette[ok], dataset.silhouette[ok])[0, 1],
                     auc_silhouette=skm.roc_auc_score(d[LABEL][ok], d.silhouette[ok]),
                     precision=np.median(r[:, 0]), recall=np.median(r[:, 1]), AP=np.median(r[:, 2]),
                     dP=dd[:, 0].mean(), dP_ci=ci(dd[:, 0]), dR=dd[:, 1].mean(), dR_ci=ci(dd[:, 1]), dAP=dd[:, 2].mean(), dAP_ci=ci(dd[:, 2])))
res = pd.DataFrame(rows).set_index("variant").round(3)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(res)
print("failed variants:", ERR)

,corr_silhouette,auc_silhouette,precision,recall,AP,dP,dP_ci,dR,dR_ci,dAP,dAP_ci
variant,,,,,,,,,,,
baseline (current features),1.000,0.633,0.917,0.591,0.901,0.000,0.000,0.000,0.000,0.000,0.000
baseline settings (check),1.000,0.633,0.917,0.591,0.901,0.000,0.000,0.000,0.000,0.000,0.000
SVD denoising rank 2,0.945,0.640,0.913,0.575,0.897,-0.002,0.003,0.002,0.005,-0.002,0.002
SVD denoising rank 3,0.962,0.648,0.917,0.575,0.895,-0.001,0.003,0.001,0.004,-0.001,0.002
SVD denoising rank 5,0.969,0.646,0.917,0.591,0.897,-0.002,0.004,0.001,0.004,-0.002,0.002
amplitude-normalized waveforms (silhouette only),0.858,0.665,0.917,0.586,0.897,-0.003,0.009,-0.010,0.009,-0.003,0.003
trough-aligned spikes,0.967,0.626,0.923,0.591,0.896,0.010,0.006,0.002,0.005,0.000,0.002
PCA on the 1 main channel(s),0.523,0.683,0.900,0.375,0.748,-0.015,0.017,-0.215,0.018,-0.146,0.011
PCA on the 3 main channel(s),0.682,0.638,0.920,0.591,0.880,-0.001,0.009,0.004,0.010,-0.010,0.006


failed variants: {'PCA mode concatenated (6 main channels)': "IndexError('too many indices for array: array is 2-dimensional, but 3 were indexed')"}
